# LTS PDR (based on Kori et al. 2023)

The example from Figure 1.

In [1]:
S = {0,1,2,3,4,5,6}
delta = {0: {1,2},
         1: {0,3},
         2: {2,3},
         3: {4},
         4: {4},
         5: {5,6},
         6: {5,6}}
I = {0}
P = {0,1,2,3,4,5}

F and G are adjoints, because $F(x) \subseteq y \iff x \subseteq G(y)$.

In [3]:
import itertools
def F(X):
    return set(itertools.chain.from_iterable([delta[x] for x in X]))
def G(X):
    return {s for s in S if delta[s] <= X}
def F_I(X):
    return I | F(X)
def G_P(X):
    return P & G(X)

def ntimes(f, x, n):
    """Apply a function f times to its argument"""
    res = x
    for _ in range(n):
        res = f(res)
    return res

pos_chain = [ntimes(F_I, set(), n) for n in range(6)]
print(pos_chain)
neg_chain = [ntimes(G_P, S, n) for n in range(6)]
print(neg_chain)

[set(), {0}, {0, 1, 2}, {0, 1, 2, 3}, {0, 1, 2, 3, 4}, {0, 1, 2, 3, 4}]
[{0, 1, 2, 3, 4, 5, 6}, {0, 1, 2, 3, 4, 5}, {0, 1, 2, 3, 4}, {0, 1, 2, 3, 4}, {0, 1, 2, 3, 4}, {0, 1, 2, 3, 4}]


In [5]:
def is_pos_conclusive(x):
    if len(x) < 2:
        return False
    for j in range(len(x)-1):
        if x[j] == x[j+1]:
            return True
    return False

def is_neg_conclusive(i, y):
    for l in y:
        if not i <= l:
            return True

def is_conclusive(i, x, y):
    return is_pos_conclusive(x) or is_neg_conclusive(i,y)

def init_neg_heuristic(p): # candidate
    return p

def extend_neg_heuristic(g, y): # decide
    return g(y[0])

def tighten_pos_heuristic_final(f, i, x, y, k): # conflict
    return y[0]

def tighten_pos_heurisitc_initial(f, i, x, y, k):
    return i | f(x[k-1])

MAX_ITERS = 30

def adjPDR(i, f, g, p, top, bottom, join, meet):
    iteration = 0
    x = [bottom, top]
    y = []
    while not is_conclusive(i,x,y):
        n = len(x)
        k = n - len(y) # Careful, the indices for y are offset by k!
        iteration += 1

        print(f"\n{iteration}")
        print(f"n: {n}, k: {k}")

        if iteration >= MAX_ITERS:
            print("Loopy")
            return None

        if y == [] and x[n-1] <= p:
            print("Extend-Pos / Unfold")
            x.append(top)
            
        elif y == [] and not x[n-1] <= p:
            print("Init-Neg / Candidate")
            z = init_neg_heuristic(p)
            y.append(z)
            
        elif y != [] and not f(x[k-1]) <= g(y[0]): # k offset
            print("Extend-Neg / Decide")
            z = extend_neg_heuristic(g, y)
            y.insert(0,z)
            
        elif y != [] and f(x[k-1]) <= y[0]:
            print("Tighten-Pos / Conflict")
            z = tighten_pos_heurisitc_initial(f, i, x, y, k)
            x = [meet(xj, z) for (j, xj) in enumerate(x) if j <= k] + [xj for j in range(k+1, n-1)]
            y.pop(0)
        else:
            print("NOTHING???")
        print("x", x)
        print("y", y)

    if is_pos_conclusive(x):
        return True
    else:
        return False

join = lambda x, y : x | y # union
meet = lambda x, y : x & y # intersection

adjPDR(I, F, G, P, S, set(), join, meet)


1
n: 2, k: 2
Init-Neg / Candidate
x [set(), {0, 1, 2, 3, 4, 5, 6}]
y [{0, 1, 2, 3, 4, 5}]

2
n: 2, k: 1
Tighten-Pos / Conflict
x [set(), {0}]
y []

3
n: 2, k: 2
Extend-Pos / Unfold
x [set(), {0}, {0, 1, 2, 3, 4, 5, 6}]
y []

4
n: 3, k: 3
Init-Neg / Candidate
x [set(), {0}, {0, 1, 2, 3, 4, 5, 6}]
y [{0, 1, 2, 3, 4, 5}]

5
n: 3, k: 2
Tighten-Pos / Conflict
x [set(), {0}, {0, 1, 2}]
y []

6
n: 3, k: 3
Extend-Pos / Unfold
x [set(), {0}, {0, 1, 2}, {0, 1, 2, 3, 4, 5, 6}]
y []

7
n: 4, k: 4
Init-Neg / Candidate
x [set(), {0}, {0, 1, 2}, {0, 1, 2, 3, 4, 5, 6}]
y [{0, 1, 2, 3, 4, 5}]

8
n: 4, k: 3
Tighten-Pos / Conflict
x [set(), {0}, {0, 1, 2}, {0, 1, 2, 3}]
y []

9
n: 4, k: 4
Extend-Pos / Unfold
x [set(), {0}, {0, 1, 2}, {0, 1, 2, 3}, {0, 1, 2, 3, 4, 5, 6}]
y []

10
n: 5, k: 5
Init-Neg / Candidate
x [set(), {0}, {0, 1, 2}, {0, 1, 2, 3}, {0, 1, 2, 3, 4, 5, 6}]
y [{0, 1, 2, 3, 4, 5}]

11
n: 5, k: 4
Tighten-Pos / Conflict
x [set(), {0}, {0, 1, 2}, {0, 1, 2, 3}, {0, 1, 2, 3, 4}]
y []

12
n: 5, 

True